In [ ]:
import pathlib
import random
import copy
import numpy as np
import torch

from captum.attr import *
import matplotlib.pyplot as plt
from omegaconf import OmegaConf
import yaml
import argparse
from data.data_loader import load_eeg_data, get_sliding_window_data, create_dataloader
import os
import mne
from tqdm import tqdm
from sklearn.preprocessing import MinMaxScaler
import pandas as pd
import quantus

from models.s4net import S4PatchedFinalNet,TrunkNet, HeadNet
import gc


from sklearn.metrics.pairwise import cosine_similarity, euclidean_distances
from scipy.spatial.distance import mahalanobis
import itertools
from scipy.linalg import sqrtm
from scipy.cluster.hierarchy import dendrogram, linkage
import matplotlib.pyplot as plt


In [ ]:
CFG_YAML = """
wandb:
 key: f0c92a0059bf12e2647f0a1c22fdcd12555fa6df
model:
dataset:
 data_directory: /home/marco/Documents/GitHub/tms_eeg_decoding/data
 #file_name: subject_{:03d}_preprocessed_combined_py.fif
 file_name: subject_{:03d}_preprocessed_combined_py.fif
 exclude_timepoints: 100
 subject_index: 1
 test_subject_indices: [1,2,13,24,26,27,29,34,35,41, 42,43,45,46,47,48,52,55,56,57,60,62,67,69,72,73,79,80,86,88,92,102]
 #test_subject_indices: [2]
training:
 training_start_len: 100
 pretrain_epochs: 100
 pretrain_lr: 0.0001
 val_window_len: 1
 epochs_per_window: 10
 num_warmup_epochs: 5
 num_epochs: 800
 slide_step: 1
 num_warmup_epochs_per_window: 0
 lr: 0.005 #maybe change back to 0.0001
 nll_beta: 0.001
 num_warmup_epochs: 0
 batch_size: 50 #better to use 50
 random_seed: 42
 precision: bf16
 kde_lambda: 0.5
 finetune_entire_model: true # Set to true to finetune the entire model, false for transformer only
exp_name: S4_S4EEGNet_ema
"""

def load_config():
    cfg = OmegaConf.create(yaml.safe_load(CFG_YAML))
    cfg.exp_name = f"{cfg.exp_name}_subject_{cfg.dataset.subject_index}"
    return cfg

def parse_args():
    parser = argparse.ArgumentParser()
    parser.add_argument("--update_conf", nargs="*", help="Updates to the configuration in the form of key=value pairs", default=[])
    parser.add_argument("-f", "--fff", help="A dummy argument to handle IPython's default argument", default="1")
    return parser.parse_args()

def update_config(cfg, cli_args):
    for update in cli_args.update_conf:
        key, value = update.split("=")
        try:
            value = eval(value)
        except:
            pass
        OmegaConf.update(cfg, key, value, force_add=True)
    cfg.exp_name = cfg.exp_name + "_" + "_".join(cli_args.update_conf)
    print(OmegaConf.to_yaml(cfg))
    return cfg


def save_config(cfg):
    os.makedirs("conf/sweeps", exist_ok=True)
    os.makedirs("exp/withinsubs", exist_ok=True)
    with open(f"conf/sweeps/withinsubs_{cfg.exp_name}.yaml", "w") as f:
        f.write(OmegaConf.to_yaml(cfg))

In [ ]:
def load_data_subject(subject_index=2):
    data_directory_power = "/home/marco/Documents/GitHub/tms_eeg_decoding/subject_clustering/power_perturbation_results"
    data_directory_phase = "/home/marco/Documents/GitHub/tms_eeg_decoding/subject_clustering/phase_perturbation_results"
    power_data = np.load(f"{data_directory_power}/power_perturbation_results_{subject_index}.npy", allow_pickle=True)
    phase_data = np.load(f"{data_directory_phase}/phase_perturbation_results_{subject_index}.npy", allow_pickle=True)
    power_values = np.array(list(power_data .item().values()))
    phase_values = np.array(list(phase_data.item().values()))
    combined_values = np.concatenate((power_values, phase_values))
    return combined_values
  

In [ ]:
cfg = load_config()
all_data = np.zeros((len(cfg.dataset.test_subject_indices), 60))
for si, subject_index in enumerate(cfg.dataset.test_subject_indices):
    all_data[si] = load_data_subject(subject_index)


In [ ]:
colors = [
    "#1f77b4", "#ff7f0e", "#2ca02c", "#d62728", "#9467bd", "#8c564b", "#e377c2", "#7f7f7f", "#bcbd22", "#17becf",
    "#ff6699", "#66ff66", "#ffcc00", "#3399ff", "#9933ff", "#ff3300", "#33cccc", "#ff9900", "#009933", "#cc3399",
    "#0000ff", "#ff0066", "#00ffcc", "#6600cc", "#ffcc99", "#336600", "#ffccff", "#00cc66", "#cc9900", "#663399",
    "#999999", "#ff5050"]


In [ ]:
from sklearn.manifold import TSNE
import umap

# Perform t-SNE
tsne = TSNE(n_components=2)
all_data_tsne = tsne.fit_transform(all_data)

# Perform UMAP
umap_model = umap.UMAP(n_components=2)
all_data_umap = umap_model.fit_transform(all_data)

# Plotting the results
plt.figure(figsize=(12, 6))

plt.subplot(1, 2, 1)
plt.scatter(all_data_tsne[:, 0], all_data_tsne[:, 1], c=colors)
for i, txt in enumerate(cfg.dataset.test_subject_indices):
    plt.annotate(txt, (all_data_tsne[i, 0], all_data_tsne[i, 1]), fontsize=8)
plt.title('t-SNE')

plt.subplot(1, 2, 2)
plt.scatter(all_data_umap[:, 0], all_data_umap[:, 1], c=colors)
for i, txt in enumerate(cfg.dataset.test_subject_indices):
    plt.annotate(txt, (all_data_umap[i, 0], all_data_umap[i, 1]), fontsize=8)
plt.title('UMAP')

plt.savefig(f"cluster_by_perturbation_results_UMAP_TSNE.png")
plt.show()

TSNE is likely the better way to visualize this as preserving the local neighborhood is more important as we only want to know which subjects are in a cluster, is matters not as much how far these clusters are apart

# cluster based on precomputed pairwise distances

In [ ]:
from scipy.spatial import distance

# Compute the pairwise Euclidean distances
euclidean_distances = distance.cdist(all_data, all_data, metric='euclidean')

print(euclidean_distances.shape)
plt.figure(figsize=(10, 8))
plt.imshow(1-euclidean_distances, cmap='viridis', interpolation='nearest')
plt.colorbar()
plt.title('Pairwise Euclidean Distances')
plt.xlabel('Subject Index')
plt.ylabel('Subject Index')
plt.show()


In [ ]:
all_data.shape

# plot by pre-computed pairwise distances

In [ ]:
def compute_fid(activatoins1, activations2):

    mu1, sigma1 = np.mean(activatoins1, axis=0), np.cov(activatoins1, rowvar=False)
    mu2, sigma2 = np.mean(activations2, axis=0), np.cov(activations2, rowvar=False)

    diff = mu1 - mu2
    covmean, _ = sqrtm(sigma1.dot(sigma2), disp=False)

    # Numerical stability check
    if np.iscomplexobj(covmean):
        covmean = covmean.real

    fid = diff.dot(diff) + np.trace(sigma1 + sigma2 - 2 * covmean)
    return fid

In [ ]:
def compute_euclidean_and_fid(activation1, activation2):
    # Ensure both activations have the same number of samples by subsampling
    min_samples = min(activation1.shape[0], activation2.shape[0])
    indices1 = np.random.choice(activation1.shape[0], min_samples, replace=False)
    indices2 = np.random.choice(activation2.shape[0], min_samples, replace=False)

    activation1_subsampled = activation1[indices1]
    activation2_subsampled = activation2[indices2]

    # Compute the Euclidean distance between the subsampled activations
    euclidean_distance = np.linalg.norm(activation1_subsampled - activation2_subsampled)



    return euclidean_distance


In [ ]:
def compute_mahalanobis_distance(activation1, activation2):
    # Ensure both activations have the same number of samples by subsampling
    min_samples = min(activation1.shape[0], activation2.shape[0])
    indices1 = np.random.choice(activation1.shape[0], min_samples, replace=False)
    indices2 = np.random.choice(activation2.shape[0], min_samples, replace=False)

    activation1_subsampled = activation1[indices1]
    activation2_subsampled = activation2[indices2]

    

    # Compute the mean and covariance of the activations for both subjects
    mean1 = np.mean(activation1_subsampled, axis=0)
    mean2 = np.mean(activation2_subsampled, axis=0)
    cov1 = np.cov(activation1_subsampled, rowvar=False)
    cov2 = np.cov(activation2_subsampled, rowvar=False)

    # Compute the pooled covariance matrix
    pooled_cov = (cov1 + cov2) / 2

    # Compute the Mahalanobis distance
    inv_pooled_cov = np.linalg.inv(pooled_cov)
    mahalanobis_dist = mahalanobis(mean1, mean2, inv_pooled_cov)

    return mahalanobis_dist

In [ ]:
def compute_cosine_distance(activation1, activation2):
    """
    Compute the cosine distance between two activations.

    Parameters:
    activation1 (numpy.ndarray or torch.Tensor): The first activation.
    activation2 (numpy.ndarray or torch.Tensor): The second activation.

    Returns:
    float: The average cosine distance between the two activations.
    """
    if isinstance(activation1, torch.Tensor):
        activation1 = activation1.cpu().numpy()
    if isinstance(activation2, torch.Tensor):
        activation2 = activation2.cpu().numpy()

    min_samples = min(activation1.shape[0], activation2.shape[0])
    indices1 = np.random.choice(activation1.shape[0], min_samples, replace=False)
    indices2 = np.random.choice(activation2.shape[0], min_samples, replace=False)

    activation1_subsampled = activation1[indices1]
    activation2_subsampled = activation2[indices2]
    
    cosine_sim = cosine_similarity(activation1_subsampled, activation2_subsampled)
    cosine_distance = 1 - np.mean(np.diag(cosine_sim))
    
    return cosine_distance

In [ ]:
from sklearn.manifold import MDS
from sklearn.decomposition import PCA

def plot_mds_and_pca(subject_indices, frobenius_distances, name="Frobenius"):
    # Perform MDS on the Frobenius distances
    mds = MDS(n_components=2, dissimilarity="precomputed", random_state=42)
    points_2d = mds.fit_transform(frobenius_distances)

    # Perform PCA on the Frobenius distances
    pca = PCA(n_components=2)
    points_pca = pca.fit_transform(frobenius_distances)

    # Plot the PCA results
    plt.figure(figsize=(9, 3))
    for i, subject in enumerate(subject_indices):
        plt.scatter(points_pca[i, 0], points_pca[i, 1], label=f'Subject {subject}')
        plt.text(points_pca[i, 0], points_pca[i, 1], str(subject), fontsize=9)
    #plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
    plt.title(f'PCA Plot of Subjects based on {name} Distances')
    plt.xlabel('PCA Dimension 1')
    plt.ylabel('PCA Dimension 2')
    plt.savefig(f'pairwise_distances_last_layer/{name}_pca.png')
    plt.show()

    # Plot the MDS results
    plt.figure(figsize=(9, 3))
    for i, subject in enumerate(subject_indices):
        plt.scatter(points_2d[i, 0], points_2d[i, 1], label=f'Subject {subject}')
        plt.text(points_2d[i, 0], points_2d[i, 1], str(subject), fontsize=9)
    #plt.legend(bbox_to_anchor=(0.5, -0.1), loc='upper center', ncol=1)
    plt.title(f'MDS Plot of Subjects based on {name} Norm Distances')
    plt.xlabel('MDS Dimension 1')
    plt.ylabel('MDS Dimension 2')
    plt.savefig(f'pairwise_distances_perturbation/{name}_mds.png')
    plt.show()

# Example usage:
#plot_mds_and_pca(subject_indices, frobenius_distances)

In [ ]:
import itertools
def compute_pairwise_distances(data,  subject_indices):
    num_subjects = len(subject_indices)
    pairings = list(itertools.combinations(np.arange(num_subjects), 2))
    #results = []
    # would be more efficient to load the data once and then extract the activations for all pairs
    # but memory might be an issue
    dir = "pairwise_distances_perturbation"
    results = {}
    # compute and store actiations for all subjects
    activations = {}

    for pair in tqdm(pairings):
        #print(data[pair[0]])
        #print(data[pair[1]])
        euclidean_distance = compute_euclidean_and_fid(data[pair[0]], data[pair[1]])
        #mahalanobis_dist = compute_mahalanobis_distance(data[pair[0]], data[pair[1]])
        cosine_distance = compute_cosine_distance(data[pair[0]].reshape(1,-1), data[pair[1]].reshape(1,-1))
        result = {"pair": (subject_indices[int(pair[0])],subject_indices[int(pair[1])]) , "euclidean_dist" : euclidean_distance, "cosine_distance": cosine_distance}
        #result =  {"pair": pair, "frobenius_norm": frobenius_norm, "fid": fid, "mahalanobis_dist": mahalanobis_dist}
        #results.append((pair, frobenius_norm, fid))
        #print(f"Pair: {pair}, Frobenius norm: {frobenius_norm:.3f}, FID: {fid:.3f}")
        print(result)
        os.makedirs(dir, exist_ok=True)
        np.save(os.path.join(dir,f"results_{subject_indices[int(pair[0])]}_{subject_indices[int(pair[1])]}.npy",), result)
        results[f"{subject_indices[int(pair[0])]}_{subject_indices[int(pair[1])]}"] = result
    return results




In [ ]:
results = compute_pairwise_distances(all_data, cfg.dataset.test_subject_indices)

In [ ]:
euclidean_distances_dict = {key: value['euclidean_dist'] for key, value in results.items()}

In [ ]:
def plot_dendrogram(subject_indices, distances, name="Frobenius"):
    # Extract distances from the results


    # Treat distances as a distance matrix by taking the upper triangular matrix and flattening it
    triu_distances = distances[np.triu_indices(len(subject_indices), k=1)]
    # scale distances but only allow positive values
    #triu_distances = np.clip(triu_distances, 0, None)
    #triu_distances = MinMaxScaler().fit_transform(triu_distances.reshape(-1, 1)).flatten()
    # Compute the linkage matrix
    
    
    Z = linkage(triu_distances)

    # Plot the dendrogram
    plt.figure(figsize=(9, 3))
    dendrogram(Z, labels=subject_indices, leaf_rotation=90)
    plt.title(f'Hierarchical Clustering Dendrogram ({name} Distance)')
    plt.xlabel('Subject Index')
    plt.ylabel(f'{name} Distance')
    plt.savefig(f'pairwise_distances_perturbation/{name}_dendrogram.png')
    plt.show()

# Example usage:
#plot_dendrogram(subject_indices, frobenius_distances)

In [ ]:
subject_indices = cfg.dataset.test_subject_indices 
pairings = list(itertools.combinations(subject_indices, 2))
euclidean_distances = np.zeros((len(subject_indices), len(subject_indices)))
for pair in pairings:
    distance = results[f'{pair[0]}_{pair[1]}']['euclidean_dist']
    i, j = subject_indices.index(pair[0]), subject_indices.index(pair[1])
    euclidean_distances[i, j] = distance
    euclidean_distances[j, i] = distance

In [ ]:
plot_dendrogram(subject_indices, euclidean_distances, name="Euclidean")

In [ ]:
def plot_mds_and_pca(subject_indices, distances, name="Frobenius"):
    # Perform MDS on the Frobenius distances
    mds = MDS(n_components=2, dissimilarity="precomputed", random_state=42)
    points_2d = mds.fit_transform(distances)

    # Perform PCA on the Frobenius distances
    pca = PCA(n_components=2)
    points_pca = pca.fit_transform(distances)

    # Plot the PCA results
    plt.figure(figsize=(9, 3))
    for i, subject in enumerate(subject_indices):
        plt.scatter(points_pca[i, 0], points_pca[i, 1], label=f'Subject {subject}')
        plt.text(points_pca[i, 0], points_pca[i, 1], str(subject), fontsize=9)
    #plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
    plt.title(f'PCA Plot of Subjects based on {name} Distances')
    plt.xlabel('PCA Dimension 1')
    plt.ylabel('PCA Dimension 2')
    plt.savefig(f'pairwise_distances_perturbation/{name}_pca.png')
    plt.show()

    # Plot the MDS results
    plt.figure(figsize=(9, 3))
    for i, subject in enumerate(subject_indices):
        plt.scatter(points_2d[i, 0], points_2d[i, 1], label=f'Subject {subject}')
        plt.text(points_2d[i, 0], points_2d[i, 1], str(subject), fontsize=9)
    #plt.legend(bbox_to_anchor=(0.5, -0.1), loc='upper center', ncol=1)
    plt.title(f'MDS Plot of Subjects based on {name} Norm Distances')
    plt.xlabel('MDS Dimension 1')
    plt.ylabel('MDS Dimension 2')
    plt.savefig(f'pairwise_distances_peturbation/{name}_mds.png')
    plt.show()

# Example usage:
#plot_mds_and_pca(subject_indices, frobenius_distances)

In [ ]:
plot_mds_and_pca(subject_indices, euclidean_distances, name="Euclidean")